<div style="background-color: black;">
    <hr style="border: 4px solid skyblue;">
<div style="text-align: center; margin-left: 0em; font-weight: bold; font-size: 20px; font-family: TimesNewRoman; color: skyblue;">
    AVAILABILITY FACTORS DATA PROCESSING
    <br>
    SOLAR | WIND OFFSHORE | WIND ONSHORE | RUN OF RIVER
</div>
<div style="text-align: center; margin-left: 0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    Main Formatting Notebook
    <br>
    from PYPSA to DISPA-SET
</div>
<br>
<div style="text-align: justify; margin-left: 0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
This script is used to process the raw time series data for renewable source availability factors obtained from the PyPSA model simulations conducted for the Dispa-SET Unleashed project.
    <br>
The explanation text cells detail the entire process and help the reader understand how the final results were achieved step-by-step
</div>
<br>
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    1. Notebook Set Up
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Importing needed libraries.
<hr style="border: 1px solid skyblue;">
</div>
</div>

In [1]:
import os
import csv
from datetime import datetime
import requests
import pandas as pd
from shutil import move
import numpy as np
import shutil
from bs4 import BeautifulSoup
import re
import io
import plotly.graph_objects as go
from typing import List, Dict, Tuple
import re
from IPython.display import HTML
from difflib import get_close_matches
from collections import defaultdict

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Auxiliar Code
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cell has the purpose to create the correponding folders with the name of all the EU countries available in the ENTSOE data base.
    <br>
    Uncomment it to use it just if needed.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [2]:
"""
# List of countries with their acronyms in parentheses
countries = [
    "Albania       (AL)", "Armenia        (AM)", "Austria          (AT)", "Azerbaijan (AZ)",
    "Belarus       (BY)", "Belgium        (BE)", "Bosnia and Herz. (BA)", "Bulgaria   (BG)",
    "Croatia       (HR)", "Cyprus         (CY)", "Czech Republic   (CZ)", "Denmark    (DK)",
    "Estonia       (EE)", "Finland        (FI)", "France           (FR)", "Georgia    (GE)",
    "Germany       (DE)", "Greece         (EL)", "Hungary          (HU)", "Iceland    (IS)",
    "Ireland       (IE)", "Italy          (IT)", "Kosovo           (XK)", "Latvia     (LV)",
    "Lithuania     (LT)", "Luxembourg     (LU)", "Malta            (MT)", "Moldova    (MD)",
    "Montenegro    (ME)", "Netherlands    (NL)", "North Macedonia  (MK)", "Norway     (NO)",
    "Poland        (PL)", "Portugal       (PT)", "Romania          (RO)", "Russia     (RU)",
    "Russia Legacy (RU)", "Serbia         (RS)", "Slovakia         (SK)", "Slovenia   (SI)",
    "Spain         (ES)", "Sweden         (SE)", "Switzerland      (CH)", "Turkey     (TR)",
    "Ukraine       (UA)", "United Kingdom (UK)"
]

# Set the path where you want to create the folders
base_path = '/home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/AvailabilityFactors'

# Ensure the base path exists
os.makedirs(base_path, exist_ok=True)

# Loop through the list of countries
for country_string in countries:
    # Use a regular expression to find the acronym inside the parentheses
    match = re.search(r'\((.*?)\)', country_string)

    # If a match is found, extract the acronym
    if match:
        acronym = match.group(1).strip()  # Use strip() to remove any extra whitespace
        folder_path = os.path.join(base_path, acronym)

        # Check if the folder already exists to avoid errors
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)
            print(f"Created folder: {folder_path}")
        else:
            print(f"Folder already exists: {folder_path}")
    else:
        print(f"Could not extract acronym from: {country_string}")

print("\nAll folders created successfully!")
"""

<>:1: SyntaxWarning: invalid escape sequence '\('
<>:1: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_1139130/3620205441.py:1: SyntaxWarning: invalid escape sequence '\('
  """


'\n# List of countries with their acronyms in parentheses\ncountries = [\n    "Albania       (AL)", "Armenia        (AM)", "Austria          (AT)", "Azerbaijan (AZ)",\n    "Belarus       (BY)", "Belgium        (BE)", "Bosnia and Herz. (BA)", "Bulgaria   (BG)",\n    "Croatia       (HR)", "Cyprus         (CY)", "Czech Republic   (CZ)", "Denmark    (DK)",\n    "Estonia       (EE)", "Finland        (FI)", "France           (FR)", "Georgia    (GE)",\n    "Germany       (DE)", "Greece         (EL)", "Hungary          (HU)", "Iceland    (IS)",\n    "Ireland       (IE)", "Italy          (IT)", "Kosovo           (XK)", "Latvia     (LV)",\n    "Lithuania     (LT)", "Luxembourg     (LU)", "Malta            (MT)", "Moldova    (MD)",\n    "Montenegro    (ME)", "Netherlands    (NL)", "North Macedonia  (MK)", "Norway     (NO)",\n    "Poland        (PL)", "Portugal       (PT)", "Romania          (RO)", "Russia     (RU)",\n    "Russia Legacy (RU)", "Serbia         (RS)", "Slovakia         (SK)", "Slove

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    2. Dispa-SET_Unleash Folder Path
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Determinning dynamically the zone_folder_path based on the location of the "Dispa-SET_Unleash" folder relative to the current working directory. 
<br>
    If the "Dispa-SET_Unleash" folder is copied to a different machine or location, the dispaSET_unleash_folder_path variable will automatically adjust accordingly.
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [3]:
# Get the current working directory
current_directory = os.getcwd()

# Navigate to the parent directory of "Dispa-SET_Unleash"
dispaSET_unleash_parent_directory = os.path.dirname(current_directory)

# Get the path to the "Dispa-SET_Unleash" folder
dispaSET_unleash_folder_path = os.path.dirname(dispaSET_unleash_parent_directory)

# Construct the dispaSET_unleash_folder_name variable
dispaSET_unleash_folder_name = os.path.basename(dispaSET_unleash_folder_path)

print("dispaSET_unleash_folder_name:", dispaSET_unleash_folder_name)
print("dispaSET_unleash_folder_path:", dispaSET_unleash_folder_path)

dispaSET_unleash_folder_name: Dispa-SET_Unleash
dispaSET_unleash_folder_path: /home/ray/Dispa-SET_Unleash


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
2.1. PyPSA Source Scenario
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
There are two sccenarios as source of PyPSA power plants raw data.
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman; color:skyblue">
<li>
Reference_Scenario
<li>
Suficiency_Scenario
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The scenario variable must be selected before proceeding to the next processing steps
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [4]:
#pypsa_scenario = "Reference_Scenario"
pypsa_scenario = "Suficiency_Scenario"

print("PyPSA Chosen Scenario:", pypsa_scenario)

PyPSA Chosen Scenario: Suficiency_Scenario


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
2.2. Secondary Folders Path
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The many subfolders within the Unleash directory require their location paths to be defined for correct access during processing.
<br>
All of these are dependent on the chosen scenario.
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [5]:
# Get the path to the "Dispa-SET_Unleash_Availability_Factor" folder of the data used as base
additional_path_1 = "/Database/AvailabilityFactors"
availability_factors_base_data_folder_path = dispaSET_unleash_folder_path + additional_path_1

# Construct the Dispa-SET_Unleash_Availability_Factor_name variable of the data used as reference
availability_factors_base_data_folder_name = os.path.basename(availability_factors_base_data_folder_path)

print("availability_factors_base_data_folder_name:", availability_factors_base_data_folder_name)
print("availability_factors_base_data_folder_path:", availability_factors_base_data_folder_path)
# --------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Get the path to the "Dispa-SET_Unleash_Availability_Factor" folder of the PyPSA raw data
additional_path_2 = os.path.join("RawData_PyPSA", pypsa_scenario, "dispatch")

availability_factors_pypsa_raw_data_folder_path = dispaSET_unleash_folder_path + '/' + additional_path_2

# Construct the Dispa-SET_Unleash_Availability_Factor_folder_name variable of the PyPSA raw data
availability_factors_pypsa_raw_data_folder_name = os.path.basename(availability_factors_pypsa_raw_data_folder_path)

print("availability_factors_pypsa_raw_data_folder_name:", availability_factors_pypsa_raw_data_folder_name)
print("availability_factors_pypsa_raw_data_folder_path:", availability_factors_pypsa_raw_data_folder_path)
# --------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Get the path to the "Dispa-SET_Unleash_Availability_Factor" folder of the PyPSA formated data
additional_path_3 = os.path.join("Database_PyPSA", pypsa_scenario, "AvailabilityFactors")

availability_factors_pypsa_formated_data_folder_path = dispaSET_unleash_folder_path + '/' + additional_path_3

# Construct the Dispa-SET_Unleash_Availability_Factor_folder_name variable of the PyPSA formated data
availability_factors_pypsa_formated_data_folder_name = os.path.basename(availability_factors_pypsa_formated_data_folder_path)

print("availability_factors_pypsa_formated_data_folder_name:", availability_factors_pypsa_formated_data_folder_name)
print("availability_factors_pypsa_formated_data_folder_path:", availability_factors_pypsa_formated_data_folder_path)
# --------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Get the path to the "Dispa-SET_Unleash_Power_Plants" folder of the PyPSA formated data
additional_path_4 = os.path.join("Database_PyPSA", pypsa_scenario, "PowerPlants")

power_plants_pypsa_formated_data_folder_path = dispaSET_unleash_folder_path + '/' + additional_path_4

# Construct the dispaSET_unleash_power_plants_folder_name variable of the PyPSA formated data
power_plants_pypsa_formated_data_folder_name = os.path.basename(power_plants_pypsa_formated_data_folder_path)

print("power_plants_pypsa_formated_data_folder_name:", power_plants_pypsa_formated_data_folder_name)
print("power_plants_pypsa_formated_data_folder_path:", power_plants_pypsa_formated_data_folder_path)

availability_factors_base_data_folder_name: AvailabilityFactors
availability_factors_base_data_folder_path: /home/ray/Dispa-SET_Unleash/Database/AvailabilityFactors
availability_factors_pypsa_raw_data_folder_name: dispatch
availability_factors_pypsa_raw_data_folder_path: /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Suficiency_Scenario/dispatch
availability_factors_pypsa_formated_data_folder_name: AvailabilityFactors
availability_factors_pypsa_formated_data_folder_path: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/AvailabilityFactors
power_plants_pypsa_formated_data_folder_name: PowerPlants
power_plants_pypsa_formated_data_folder_path: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/PowerPlants


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    3. Zone(s) Creation
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Entering the zone name or names (comment those ones that are not available data) where all data related to the corresponding zone are going to be storage
<br>
For European country names use the ISO 3166-1 standars i.e. AT, BE, BG, CH.... etc. to give the zone_name.
</div>
<hr style="border: 1px solid skyblue;">

In [6]:
# List of folder names to be addressed
zone_names = [
                #"AL",
                #"AM",
                #"AT",
                #"AZ",
                #"BY",
                "BE",
                #"BA",
                #"BG",
                #"HR",
                #"CY",
                #"CZ",
                #"DK",
                #"EE",
                #"FI",
                "FR",
                #"GE",
                "DE",
                #"EL",
                #"HU",
                #"IS",
                #"IE",
                #"IT",
                #"XK",
                #"LV",
                #"LT",
                #"LU",
                #"MT",
                #"MD",
                #"ME",
                "NL",
                #"MK",
                #"NO",
                #"PL",
                #"PT",
                #"RO",
                #"RU",
                #"RS",
                #"SK",
                #"SI",
                #"ES",
                #"SE",
                #"CH",
                #"TR",
                #"UA",
                "UK"
             ]

print("zone_names:", zone_names)

zone_names: ['BE', 'FR', 'DE', 'NL', 'UK']


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The following dictionary specifies the possible alternative names—or synonyms/aliases—used in international nomenclature for the EU countries.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [7]:
zone_names_equivalences_dict = {

"AL"  :  {"Acronym": ["  "       ] ,   "name": ["Albania   "       ]},  
"AM"  :  {"Acronym": ["  "       ] ,   "name": ["Armenia   "       ]},
"AT"  :  {"Acronym": ["  "       ] ,   "name": ["Austria   "       ]},
"AZ"  :  {"Acronym": ["  "       ] ,   "name": ["Azerbaijan"       ]},
"BY"  :  {"Acronym": ["  "       ] ,   "name": ["Belarus"          ]},
"BE"  :  {"Acronym": ["  "       ] ,   "name": ["Belgium"          ]},
"BA"  :  {"Acronym": ["  "       ] ,   "name": ["Bosnia and Herz." ]},
"BG"  :  {"Acronym": ["  "       ] ,   "name": ["Bulgaria"         ]},
"HR"  :  {"Acronym": ["  "       ] ,   "name": ["Croatia"          ]},
"CY"  :  {"Acronym": ["  "       ] ,   "name": ["Cyprus"           ]},
"CZ"  :  {"Acronym": ["  "       ] ,   "name": ["Czech Republic"   ]},
"DK"  :  {"Acronym": ["  "       ] ,   "name": ["Denmark"          ]},
"EE"  :  {"Acronym": ["  "       ] ,   "name": ["Estonia"          ]},
"FI"  :  {"Acronym": ["  "       ] ,   "name": ["Finland"          ]},
"FR"  :  {"Acronym": ["  "       ] ,   "name": ["France"           ]},
"GE"  :  {"Acronym": ["  "       ] ,   "name": ["Georgia"          ]}, 
"DE"  :  {"Acronym": ["  "       ] ,   "name": ["Germany"          ]},
"EL"  :  {"Acronym": ["GR"       ] ,   "name": ["Greece"           ]},
"HU"  :  {"Acronym": ["  "       ] ,   "name": ["Hungary"          ]},
"IS"  :  {"Acronym": ["  "       ] ,   "name": ["Iceland"          ]},
"IE"  :  {"Acronym": ["  "       ] ,   "name": ["Ireland"          ]},
"IT"  :  {"Acronym": ["  "       ] ,   "name": ["Italy"            ]},
"XK"  :  {"Acronym": ["  "       ] ,   "name": ["Kosovo"           ]},
"LV"  :  {"Acronym": ["  "       ] ,   "name": ["Latvia"           ]},
"LT"  :  {"Acronym": ["  "       ] ,   "name": ["Lithuania"        ]},
"LU"  :  {"Acronym": ["  "       ] ,   "name": ["Luxembourg"       ]},
"MT"  :  {"Acronym": ["  "       ] ,   "name": ["Malta"            ]},
"MD"  :  {"Acronym": ["  "       ] ,   "name": ["Moldova"          ]},
"ME"  :  {"Acronym": ["  "       ] ,   "name": ["Montenegro"       ]},
"NL"  :  {"Acronym": ["  "       ] ,   "name": ["Netherlands"      ]},
"MK"  :  {"Acronym": ["  "       ] ,   "name": ["North Macedonia"  ]},
"NO"  :  {"Acronym": ["  "       ] ,   "name": ["Norway"           ]},
"PL"  :  {"Acronym": ["  "       ] ,   "name": ["Poland"           ]},
"PT"  :  {"Acronym": ["  "       ] ,   "name": ["Portugal"         ]},
"RO"  :  {"Acronym": ["  "       ] ,   "name": ["Romania"          ]},
"RU"  :  {"Acronym": ["  "       ] ,   "name": ["Russia"           ]},
"RS"  :  {"Acronym": ["  "       ] ,   "name": ["Serbia"           ]},
"SK"  :  {"Acronym": ["  "       ] ,   "name": ["Slovakia"         ]},
"SI"  :  {"Acronym": ["  "       ] ,   "name": ["Slovenia"         ]},
"ES"  :  {"Acronym": ["  "       ] ,   "name": ["Spain"            ]},
"SE"  :  {"Acronym": ["  "       ] ,   "name": ["Sweden"           ]},
"CH"  :  {"Acronym": ["  "       ] ,   "name": ["Switzerland"      ]},
"TR"  :  {"Acronym": ["  "       ] ,   "name": ["Turkey"           ]},
"UA"  :  {"Acronym": ["  "       ] ,   "name": ["Ukraine"          ]},
"UK"  :  {"Acronym": ["GB"       ] ,   "name": ["United Kingdom"   ]},
       
}

zone_names_equivalences_dict

{'AL': {'Acronym': ['  '], 'name': ['Albania   ']},
 'AM': {'Acronym': ['  '], 'name': ['Armenia   ']},
 'AT': {'Acronym': ['  '], 'name': ['Austria   ']},
 'AZ': {'Acronym': ['  '], 'name': ['Azerbaijan']},
 'BY': {'Acronym': ['  '], 'name': ['Belarus']},
 'BE': {'Acronym': ['  '], 'name': ['Belgium']},
 'BA': {'Acronym': ['  '], 'name': ['Bosnia and Herz.']},
 'BG': {'Acronym': ['  '], 'name': ['Bulgaria']},
 'HR': {'Acronym': ['  '], 'name': ['Croatia']},
 'CY': {'Acronym': ['  '], 'name': ['Cyprus']},
 'CZ': {'Acronym': ['  '], 'name': ['Czech Republic']},
 'DK': {'Acronym': ['  '], 'name': ['Denmark']},
 'EE': {'Acronym': ['  '], 'name': ['Estonia']},
 'FI': {'Acronym': ['  '], 'name': ['Finland']},
 'FR': {'Acronym': ['  '], 'name': ['France']},
 'GE': {'Acronym': ['  '], 'name': ['Georgia']},
 'DE': {'Acronym': ['  '], 'name': ['Germany']},
 'EL': {'Acronym': ['GR'], 'name': ['Greece']},
 'HU': {'Acronym': ['  '], 'name': ['Hungary']},
 'IS': {'Acronym': ['  '], 'name': ['Icelan

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    4. Data Reference Year 
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Setting the variable on the target year which formatting data is wanted to
</div>
<hr style="border: 1px solid skyblue;">

In [8]:
# Year to which data is formating to:
data_target_year = '2030'

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    5. Availability Factors Data Frame
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Creating the data frame with all the corresponding headers according the Dispa-SET nomenclature.
<br>
The dispa-SET technologies nomenclature are loaded from "<a href="https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py" style="color:skyblue">https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py</a>"
</div>
<hr style="border: 2px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
5.1. Dispa-SET Time Step
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The time series data must be resampled to a predetermined time step.
<br>
The UNLEASH project utilizes three levels of granularity
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman; color:skyblue">
<li>
One hour (1h) 
<li>
Thirty minutes (30min)
<li>
Fifteen minutes (15min)
</li>
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [9]:
# Time step to which data is formating to:
data_target_time_step = '1h'
# data_target_time_step = '15min'
# data_target_time_step = '30min'

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
5.2. Empty Zone DataFrames
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Creating empty dataframes—column headers only—for the selected zones, corresponding to the target year
<hr style="border: 1px solid skyblue;">
</div>

In [10]:
# convert data year to integer
data_target_year = int(data_target_year)  

# Dictionary to store created DataFrames
availability_factors_dfs_dict = {}

for zone in zone_names:
    zone_folder = os.path.join(availability_factors_base_data_folder_path, zone)
    if not os.path.isdir(zone_folder):
        continue  # skip if zone folder doesn't exist

    # Go into subfolder matching data_target_time_step
    timestep_folder = os.path.join(zone_folder, str(data_target_time_step))
    if not os.path.isdir(timestep_folder):
        continue  # skip if subfolder doesn't exist

    # List all csv files in the timestep folder
    csv_files = [f for f in os.listdir(timestep_folder) if f.endswith(".csv")]

    # Extract years from filenames (must be digits only)
    available_years = []
    for f in csv_files:
        name, ext = os.path.splitext(f)
        if name.isdigit():  # e.g., "2004"
            available_years.append(int(name))

    if not available_years:
        continue  # skip if no year-based CSVs exist

    # Find closest year to target
    closest_year = min(available_years, key=lambda y: abs(y - data_target_year))

    # Path to the chosen file
    chosen_file = os.path.join(timestep_folder, f"{closest_year}.csv")

    # Read only the first column
    first_col_name = pd.read_csv(chosen_file, nrows=0).columns[0]  # get first column name
    first_col_data = pd.read_csv(chosen_file, usecols=[first_col_name])

    # Read all headers
    all_headers = pd.read_csv(chosen_file, nrows=0).columns.tolist()

    # Create new DataFrame: first column has data, others are empty
    df = pd.DataFrame(columns=all_headers)
    df[first_col_name] = first_col_data[first_col_name]

    # Store in dictionary and optionally as global variable
    df_name = f"{zone}_{data_target_year}"
    availability_factors_dfs_dict[df_name] = df
    globals()[df_name] = df

    print(f"Zone {zone}: picked {closest_year}.csv for target {data_target_year} and copied first column")

print("Availability Factors Data Frames:", list(availability_factors_dfs_dict.keys()))

Zone BE: picked 2023.csv for target 2030 and copied first column
Zone FR: picked 2023.csv for target 2030 and copied first column
Zone DE: picked 2023.csv for target 2030 and copied first column
Zone NL: picked 2023.csv for target 2030 and copied first column
Zone UK: picked 2023.csv for target 2030 and copied first column
Availability Factors Data Frames: ['BE_2030', 'FR_2030', 'DE_2030', 'NL_2030', 'UK_2030']


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
5.3. Time Step Correction
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Verifying and correcting first column timestamps in each zone dataframe to match target year and selected time step.
<hr style="border: 1px solid skyblue;">
</div>

In [11]:
# Define a mapping for time step strings to pandas frequency strings
time_step_map = {
    
    "1h": "H",
    "15min": "15T",
    "30min": "30T"

}

# Get the pandas frequency string for the target time step
freq = time_step_map[data_target_time_step]

for df_name, df in availability_factors_dfs_dict.items():
    if df.empty:
        continue  # skip empty dataframes
    
    # Identify the first column
    first_col = df.columns[0]
    
    # Convert column to datetime with UTC if not already
    df[first_col] = pd.to_datetime(df[first_col], utc=True, errors='coerce')

    # Remove any rows that failed to parse
    df = df.dropna(subset=[first_col])

    # Create a date range for the correct year and time step
    start_time = pd.Timestamp(f"{data_target_year}-01-01 00:00:00", tz="UTC")
    end_time = pd.Timestamp(f"{data_target_year}-12-31 23:59:59", tz="UTC")

    correct_index = pd.date_range(start=start_time, end=end_time, freq=freq)

    # Replace the first column with the corrected date range
    if len(correct_index) >= len(df):
        df[first_col] = correct_index[:len(df)]
    else:
        # If df has more rows than the date range, extend with repeated values
        repeats = (len(df) // len(correct_index)) + 1
        df[first_col] = pd.Series(list(correct_index) * repeats)[:len(df)]

    # Update the DataFrame in the dictionary
    availability_factors_dfs_dict[df_name] = df
    globals()[df_name] = df  # optional if you use global variables

    print(f"Updated {df_name}: first column aligned with {data_target_year} and {data_target_time_step}")

Updated BE_2030: first column aligned with 2030 and 1h
Updated FR_2030: first column aligned with 2030 and 1h
Updated DE_2030: first column aligned with 2030 and 1h
Updated NL_2030: first column aligned with 2030 and 1h
Updated UK_2030: first column aligned with 2030 and 1h


/tmp/ipykernel_1139130/2452633098.py:30: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  correct_index = pd.date_range(start=start_time, end=end_time, freq=freq)
/tmp/ipykernel_1139130/2452633098.py:30: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  correct_index = pd.date_range(start=start_time, end=end_time, freq=freq)
/tmp/ipykernel_1139130/2452633098.py:30: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  correct_index = pd.date_range(start=start_time, end=end_time, freq=freq)
/tmp/ipykernel_1139130/2452633098.py:30: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  correct_index = pd.date_range(start=start_time, end=end_time, freq=freq)
/tmp/ipykernel_1139130/2452633098.py:30: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  cor

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [12]:
print (f"Name of the DispaSET Unleash folder:                          {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                          {dispaSET_unleash_folder_path}\n")
print (f"Name of the Availability Factors Base data folder:            {availability_factors_base_data_folder_name}\n")
print (f"Path to the Availability Factors Base data folder:            {availability_factors_base_data_folder_path}\n")
print (f"Name of Availability Factors_Pypsa Raw data folder:           {availability_factors_pypsa_raw_data_folder_name}\n")
print (f"Path to the Availability Factors_Pypsa Raw data folder:       {availability_factors_pypsa_raw_data_folder_path}\n")
print (f"Name of the Availability Factors_Pypsa Formated data folder:  {availability_factors_pypsa_formated_data_folder_name}\n")
print (f"Path to the Availability Factors_Pypsa Formated data folder:  {availability_factors_pypsa_formated_data_folder_path}\n")
print (f"Name of the Power Plants_Pypsa Formated data folder:          {power_plants_pypsa_formated_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Formated data folder:          {power_plants_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                            {zone_names}\n")
print (f"Name of the zone_names_equivalences_dict (dictionary):        {list(zone_names_equivalences_dict.keys())}\n")
print (f"Target year:                                                  {data_target_year}\n")
print (f"Target time step:                                             {data_target_time_step}\n")
print (f"Name of the Availability Factors DataFrames (dictionary):     {list(availability_factors_dfs_dict.keys())}\n")

Name of the DispaSET Unleash folder:                          Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                          /home/ray/Dispa-SET_Unleash

Name of the Availability Factors Base data folder:            AvailabilityFactors

Path to the Availability Factors Base data folder:            /home/ray/Dispa-SET_Unleash/Database/AvailabilityFactors

Name of Availability Factors_Pypsa Raw data folder:           dispatch

Path to the Availability Factors_Pypsa Raw data folder:       /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Suficiency_Scenario/dispatch

Name of the Availability Factors_Pypsa Formated data folder:  AvailabilityFactors

Path to the Availability Factors_Pypsa Formated data folder:  /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/AvailabilityFactors

Name of the Power Plants_Pypsa Formated data folder:          PowerPlants

Path to the Power Plants_Pypsa Formated data folder:          /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
6. Nomenclature Technology Dictionary creation
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The following dictionary provides the mapping—or cross-reference—of renewable source technology names between the PyPSA and Dispa-SET modeling frameworks.
</div>
<hr style="border: 2px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
6.1. Dispa-SET Technologies Nomenclature
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The dispa-SET technologies nomenclature are loaded from "<a href="https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py" style="color:skyblue">https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py</a>"
</div>
<hr style="border: 0.5px solid skyblue;">
</div> 

In [13]:
# Define all the Dispa-SET technology lists from the common.py script of Dispa-SET core scripts
tech_master_list =         []

tech_renewables =          ['HROR' , 'PHOT' , 'WAVE' , 'WTOF' , 'WTON' , 'SOTH']

tech_conventional =        ['HDAM' , 'COMC' , 'GTUR' , 'STUR' , 'BATS' , 'ICEN']

tech_batteries =           ['BATS']

tech_storage =             ['BATS' , 'HDAM' , 'HPHS' , 'BEVS' , 'CAES' , 'SCSP' , 'H2ST' , 'HPHSC', 'THMS']

tech_p2bs =                ['P2GS' , 'ALKE' , 'PEME' , 'SOXE' , 'P2BS' , 'PEFC' , 'DMFC' , 'ALFC' , 'PAFC' , 'MCFC' , 'SOFC' ,
                            'REFC' , 'HDAMC', 'HRORC', 'HDLZ' , 'COMCX', 'GTURX', 'ICENX', 'STURX', 'P2HT' , 'ASHP' , 'GSHP' , 
                            'HYHP' , 'WSHP' , 'REHE']

tech_bs2p =                ['BSPG']

tech_boundary_sector =     ['BSPG' , 'GETH' , 'HOBO' , 'SOTH' , 'ABHP' , 'HOBOX', 'P2BS' , 'HBBS' , 'WHEN']

# Combine all lists into a single set to get unique values
all_technologies_set = set(tech_master_list + tech_renewables + tech_conventional +
                           tech_batteries + tech_storage + tech_p2bs + tech_bs2p +
                           tech_boundary_sector)

# Convert the set back to a sorted list
all_technologies_list = sorted(list(all_technologies_set))

# Create a DataFrame with the single column
dispaSET_tech_list = pd.DataFrame(all_technologies_list, columns=['Dispa-SET Technologies'])

# Print the final DataFrame
dispaSET_tech_list

,Dispa-SET Technologies
0,ABHP
1,ALFC
2,ALKE
3,ASHP
4,BATS
5,BEVS
6,BSPG
7,CAES
8,COMC
9,COMCX


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
6.2. Dispa-SET Fuels Nomenclature
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Additionally the dispa-SET fuelss nomenclature are loaded from "<a href="https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py" style="color:skyblue">https://github.com/energy-modelling-toolkit/Dispa-SET/blob/master/dispaset/common.py</a>"
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [14]:
# Define all the Dispa-SET fuel lists from the common.py script of Dispa-SET core scripts
dispaSET_fuel_list =      [ 'AIR', 'AMO', 'BIO', 'GAS', 'HRD', 'LIG', 'NUC', 'OIL', 'PEA', 'SUN', 
                            'WAT', 'WIN', 'WST', 'OTH', 'GEO', 'HYD', 'WHT', 'ELE', 'THE', 'UNK'  ]

# Create a DataFrame with the single column
dispaSET_fuel_list = pd.DataFrame(dispaSET_fuel_list, columns=['Dispa-SET Fuels'])

# Print the final DataFrame
dispaSET_fuel_list

,Dispa-SET Fuels
0,AIR
1,AMO
2,BIO
3,GAS
4,HRD
5,LIG
6,NUC
7,OIL
8,PEA
9,SUN


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
6.2. PyPSA vs Dispaset Nomenclature
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
All those technologies from PyPSA which can be represented in Dispa-SET have to be identified.
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: 'Times New Roman', serif; color: skyblue;">
6.2.1. PyPSA Energy System Flow Diagram
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
The next chart represents graphically how all the Energy sector inside PyPSA is structured.<br>
This is used to get the equivalent diagram for Dispaset.
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_multisector_figure_1.png" 
       alt="PyPSA Multisector Flow Diagram" 
       style="max-width:35%; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: <a href="https://pypsa-eur.readthedocs.io/en/latest/" target="_blank" style="color: skyblue; text-decoration: underline;">PyPSA-Eur Documentation</a>
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: 'Times New Roman', serif; color: skyblue;">
6.2.2. Equivalent Dispa-SET Energy System Flow Diagram
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color:skyblue">
A correlation has been established between the PyPSA parameters and their corresponding Dispa-SET equivalents:
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_sectors_as_Dispaset_Flow_work_1.svg" 
       alt="Equivalent Dispa-SET Energy System Flow Diagram" 
       style="max-width: auto; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color:skyblue">
Due to feature limitations, not all elements from PyPSA can be represented in Dispa-SET.
<br>
However, for those compatible technologies, the following chart graphically illustrates how they are connected within the Dispa-SET environment logic:
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_sectors_as_Dispaset_Flow_work_Filtered_1.svg" 
       alt="Equivalent Dispa-SET Energy System Flow Diagram" 
       style="max-width: auto; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
6.2.3. Technologies & Demmands Nomenclature - PyPSA vs Dispaset 
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The PyPSA technologies and demmands nomenclature and their correlation with their homologous from Dispa-SET are described as follows:
</div>
<table style="width: 95%; margin-left: auto; margin-right: auto; border-collapse: collapse; font-family: TimesNewRoman; font-size: 12px; color: skyblue;">
  <thead>
    <tr style="background-color: #1E1E1E; color: skyblue; border-bottom: 1px solid skyblue;">
      <th style="width: 8%; padding: 8px; text-align: left; border: 1px solid #444;">PyPSA Element</th>
      <th style="width: 15%; padding: 8px; text-align: left; border: 1px solid #444;">Technology</th>
      <th style="width: 10%; padding: 8px; text-align: left; border: 1px solid #444;">Sector Classification</th>
      <th style="width: 34%; padding: 8px; text-align: left; border: 1px solid #444;">Description</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">Sector Relation</th>
      <th style="width: 15%; padding: 8px; text-align: left; border: 1px solid #444;">Dispa-SET Element</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">Element Type</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">May Modeled?</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">DC</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents the DC (HVDC) transmission network for electricity</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">NTC</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">OCGT</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Open-Cycle Gas Turbine producing electricity from gas.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">CCGT</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Combined-Cycle Gas Turbine</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">EV charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Interface between the grid and electric vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxChargingPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">V2G</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Vehicle-to-Grid---allows EVs to discharge electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">battery charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to stored energy in batteries (charging link)</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxChargingPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">BioSNG</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Produces synthetic natural gas (bio-methane)---fuel synthesis process</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">DAC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Direct Air Capture---captures CO<sub>2</sub> for storage or utilization</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Fischer-Tropsch</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts H<sub>2</sub> + CO<sub>2</sub> to liquid hydrocarbons; fuel synthesis</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> Electrolysis</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity into hydrogen cross-sector conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> Fuel Cell</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts hydrogen back to electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> pipeline</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transports hydrogen between regions or sectors</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> pipeline retrofitted</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Existing pipelines adapted for H<sub>2</sub> transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> turbine</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Generates electricity or heat from hydrogen---boundary technology</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Haber-Bosch</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts H<sub>2</sub> + N<sub>2</sub> into ammonia---chemical/fertilizer industry</td>
      <td style="padding: 8px; border: 1px solid #444;">PX2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Steam Methane Reforming---gas to hydrogen conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR with Carbon Capture---industrial hydrogen with CC</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Sabatier</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> + CO<sub>2</sub> → CH<sub>4</sub>---synthetic methane production.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture machinery oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil used in agricultural machinery---transport/fuel</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">ammonia cracker</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts ammonia back into hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">battery discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored electricity from batteries</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity distribution grid</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents distribution-level power flow---low voltage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents coal-based electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents coal-based electricity generation or conversion node</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">nuclear</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear-to-electricity conversion within the power system.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil-fired electricity generation or conversion node</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas to gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Upgrades raw biogas into pipeline-quality methane</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas to gas CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biogas upgrading with carbon capture---industrial fuel conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biomass to liquid</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts biomass into liquid fuels---synthetic fuel process.</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> sequestered</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents captured Tons of CO<sub>2</sub> / hour transported or stored</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">coal for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Use of coal as industrial feedstock/fuel</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Supplies natural gas to industrial demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for industry CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial gas use with carbon capture</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas pipeline</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transports natural gas---energy carrier infrastructure</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas pipeline new</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Expansion of natural gas transport capacity</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">kerosene for aviation</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Aviation fuel consumption---transport sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil use for land transport---transport fuel consumption</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">methanolisation</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Synthesizes methanol (CO<sub>2</sub> + H<sub>2</sub> → CH<sub>3</sub>OH)</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">naphtha for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Provides naphtha feedstock to industrial processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">process emissions</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial process CO<sub>2</sub> emissions. Tons of CO<sub>2</sub> / hour</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">process emissions CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial CO<sub>2</sub> emissions with capture — Tons of CO<sub>2</sub> / hour</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat for rural homes</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts biomass to heat---residential fuel use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns gas for heating---non-electric final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns oil for household heating---outside power generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Uses electricity directly for heating---part of demand side</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to thermal energy in storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat — part of residential heating loop</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized air-source heat pump for urban buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass-to-heat conversion for urban residential areas</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban gas boilers---distributed thermal devices</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized oil boilers for urban homes</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heating devices---boundary heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transfers electric energy to thermal storage (heat)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass-to-heat conversion for urban residential areas</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban gas boilers---distributed thermal devices</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized oil boilers for urban homes</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">link & residential rural ground heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heat using stable ground temperature</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
     <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transfers electric energy to thermal storage (heat)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges heat from thermal storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Service sector rural building heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Provides space or process heat for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns gas for heating---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural ground heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity-to-heat for service buildings---heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil heating in rural service buildings.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heating for service buildings---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat for service‐sector thermal storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat to buildings---part of the heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Local electric heat production for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass‐to‐heat conversion---end-use heating</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns oil for household heating---outside power generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil heating---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Direct electric heating for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts power to stored heat</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Releases stored thermal energy</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping methanol</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Methanol use in maritime transport---fuel demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil consumption for ships---transport sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass fuel use for industrial heat/processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for industry CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Same as above but with carbon capture</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass transport</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents biomass logistics between regions/sectors</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">FlowXmaximum & FlowXminimum</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized district heat pump---heat sector interfac.</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Combined heat + power supplying district heating</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Plant Data</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas CHP CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Gas CHP with carbon capture---district heating system</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Plant Data</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized gas heating for urban networks</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric boiler for district heating---end-use conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solid biomass CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass combined heat + power---heat boundary process</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solid biomass CHP CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Same with carbon capture---boundary sector</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat in district storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat to the district network</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Fossil fuel-based electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas–fired power generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Coal variant used for power generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
        <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil-fired electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Powerx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">onwind</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Onshore wind turbine generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind turbine generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind-ac</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind with AC connection</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind-dc</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind with DC connection</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">solar</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Utility-scale photovoltaic generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">solar rooftop</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Distributed PV connected to power grid</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">ror</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Run-of-river hydro power plant</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">uranium</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear fuel input for electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Produces heat for households (not electricity)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized solar heating for buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Solar thermal for service-sector heat demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban service-sector solar heating</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized solar thermal for district heating</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">load</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents total shredding energy</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Load Shedding</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">storage_units</td>
      <td style="padding: 8px; border: 1px solid #444;">hydro</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Conventional hydro reservoir</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">storage_units</td>
      <td style="padding: 8px; border: 1px solid #444;">PHS</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Pumped Hydro Storage, a grid-scale electricity storage technology</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">battery</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electrical energy storage — directly coupled with the grid.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
            <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">uranium</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear fuel stock for electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen storage---chemical energy carrier</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">NH<sub>3</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Ammonia storage---chemical/fertilizer or fuel vector.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomethane stock for heating or industry</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Captured CO<sub>2</sub> pool---used in synthesis or stored</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> sequestered</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Permanent CO<sub>2</sub> storage---Tons of CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>      
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> stored</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Intermediate or final CO<sub>2</sub> reservoir---Tons of CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Coal stock for industrial/fuel processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas (CH<sub>4</sub>) stock — cross-sector energy carrier</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Fuel storage for thermal use — outside grid operations</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">methanol</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Liquid fuel stock — used in transport or synthesis chains</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil (synthetic hydrocarbons) stock for transport/industrial fuels</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass stock for heating/industrial use</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Thermal storage for rural households — heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1评审44;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Distributed heat storage in urban residences</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Heat storage for rural service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Thermal storage for urban service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">District heating storage — boundary heat network</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Final oil demand in the industrial sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Residential and tertiary DH demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">District heating demand for residential and service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Residential and tertiary heat demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized residential space/water heating demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption in lighting, irrigation, machinery)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Heat energy required in agricultural processes (drying, greenhouses)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil consumption in agricultural machinery and vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">aviation oil demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Jet fuel (kerosene) demand for aviation transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity demand for rail network</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Traction electricity used by rail and metro transport systems</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity demand of residential and tertairy</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption in households and service-sector buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity use for machinery, processes, and electrified production</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">methane</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas or synthetic methane Industrial demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">hydrogen for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen demand for industrial refining, ammonia, steelmaking</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport EV</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption by electric vehicles in road transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport hydrogen demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen fuel demand for road transport (fuel-cell vehicles)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">low-temperature heat for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">(below $\sim 200^{\circ}$C), typically supplied by boilers or heat pumps</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">Non-energy</td>
      <td style="padding: 8px; border: 1px solid #444;">naphtha for non-energy</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Naphtha used as a chemical feedstock e.g., plastics, petrochemicals</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">oil to transport demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil demand for conventional land gasoline and diesel vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen demand, maritime transport fuel-cell/ combustion ships</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Conventional marine oil fuel demand (HFO, MGO) for ships</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">solid biomass</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass demand in industrial processes for heat or material use</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
  </tbody>
</table>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
6.3. Dispa-SET vs PyPSA Technologies Equivalences Dictionary
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
To facilitate data harmonization, a dictionary containing the Dispa-SET and PyPSA technology equivalences for renewable power plant units is developed, leveraging the specifications detailed in the preceding table.
</div>
<div style="text-align: justify; margin-left:2em; font-weight: unbold; font-size: 12px; font-family: TimesNewRoman; color:skyblue">
    * Notes: &nbsp;&nbsp; Keep values of the first column identical to the 'dispaSET_tech_list' list, since it depends of the Dipsa-SET core code.
    <br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
    If the list in the <code>commons.py</code> script changes, the 'tech_equivalences' list has to be updated accordingly.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [15]:
# Dictionary mapping Dispa-SET acronyms to PyPSA tech names

tech_equivalences_dict = {
    
# -----------------------
# Renewable Power Units
# -----------------------
"WTON"  :  {"tech": ["onshore wind"      ,  "onwind"   ] ,   "curt": ["onshore curtailment"    ]} ,
    
"WTOF"  :  {"tech": ["offshore wind"     ,  "offwind"  ] ,   "curt": ["offshore curtailment"   ]} ,
    
"PHOT"  :  {"tech": ["solar"                           ] ,   "curt": ["solar curtailment"      ]} ,
    
"HROR"  :  {"tech": ["hydroelectricity"  ,  "ror"      ] ,   "curt": [" "                      ]} ,
       
}

tech_equivalences_dict

{'WTON': {'tech': ['onshore wind', 'onwind'], 'curt': ['onshore curtailment']},
 'WTOF': {'tech': ['offshore wind', 'offwind'],
  'curt': ['offshore curtailment']},
 'PHOT': {'tech': ['solar'], 'curt': ['solar curtailment']},
 'HROR': {'tech': ['hydroelectricity', 'ror'], 'curt': [' ']}}

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [16]:
print (f"Name of the DispaSET Unleash folder:                          {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                          {dispaSET_unleash_folder_path}\n")
print (f"Name of the Availability Factors Base data folder:            {availability_factors_base_data_folder_name}\n")
print (f"Path to the Availability Factors Base data folder:            {availability_factors_base_data_folder_path}\n")
print (f"Name of Availability Factors_Pypsa Raw data folder:           {availability_factors_pypsa_raw_data_folder_name}\n")
print (f"Path to the Availability Factors_Pypsa Raw data folder:       {availability_factors_pypsa_raw_data_folder_path}\n")
print (f"Name of the Availability Factors_Pypsa Formated data folder:  {availability_factors_pypsa_formated_data_folder_name}\n")
print (f"Path to the Availability Factors_Pypsa Formated data folder:  {availability_factors_pypsa_formated_data_folder_path}\n")
print (f"Name of the Power Plants_Pypsa Formated data folder:          {power_plants_pypsa_formated_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Formated data folder:          {power_plants_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                            {zone_names}\n")
print (f"Name of the zone_names_equivalences_dict (dictionary):        {list(zone_names_equivalences_dict.keys())}\n")
print (f"Target year:                                                  {data_target_year}\n")
print (f"Target time step:                                             {data_target_time_step}\n")
print (f"Name of the Availability Factors DataFrames (dictionary):     {list(availability_factors_dfs_dict.keys())}\n")
print (f"Name of the Technologies Equivalences Dictionary:             tech_equivalences_dict\n")

Name of the DispaSET Unleash folder:                          Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                          /home/ray/Dispa-SET_Unleash

Name of the Availability Factors Base data folder:            AvailabilityFactors

Path to the Availability Factors Base data folder:            /home/ray/Dispa-SET_Unleash/Database/AvailabilityFactors

Name of Availability Factors_Pypsa Raw data folder:           dispatch

Path to the Availability Factors_Pypsa Raw data folder:       /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Suficiency_Scenario/dispatch

Name of the Availability Factors_Pypsa Formated data folder:  AvailabilityFactors

Path to the Availability Factors_Pypsa Formated data folder:  /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/AvailabilityFactors

Name of the Power Plants_Pypsa Formated data folder:          PowerPlants

Path to the Power Plants_Pypsa Formated data folder:          /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency

<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    7. Raw Data Uploading
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The availability factor time series will be constructed from the hourly PyPSA dispatch values for each energy carrier
    <br>
 These values will be stored in a dictionary partitioned by country.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [17]:
# Initialize the dictionary to store DataFrames
nodal_dispatch_dfs_dict = {}

# Loop through each country acronym in zone_names
for zone in zone_names:
    # Construct the subfolder path using data_target_year
    subfolder_path = os.path.join(availability_factors_pypsa_raw_data_folder_path, str(data_target_year))

    # Try the primary file name
    primary_file_name = f"{zone}_dispatch.csv"
    primary_file_path = os.path.join(subfolder_path, primary_file_name)

    # If the primary file exists, read it
    if os.path.exists(primary_file_path):
        df = pd.read_csv(primary_file_path)
        nodal_dispatch_dfs_dict[f"{zone}_dispatch_df"] = df
    else:
        # If the primary file does not exist, try alternative acronyms
        alternative_acronyms = zone_names_equivalences_dict.get(zone, {}).get("Acronym", [])
        for alt_acronym in alternative_acronyms:
            alternative_file_name = f"{alt_acronym}_dispatch.csv"
            alternative_file_path = os.path.join(subfolder_path, alternative_file_name)
            if os.path.exists(alternative_file_path):
                df = pd.read_csv(alternative_file_path)
                nodal_dispatch_dfs_dict[f"{zone}_dispatch_df"] = df
                break  # Stop after the first successful alternative
        else:
            print(f"No file found for zone: {zone} (tried: {primary_file_name}, alternatives: {[f'{a}_dispatch.csv' for a in alternative_acronyms]})")

# Now, nodal_dispatch_dfs_dict contains all the DataFrames
nodal_dispatch_dfs_dict

{'BE_dispatch_df':                Unnamed: 0      CCGT       CHP  Imports_Exports          OCGT  \
 0     2013-01-01 00:00:00  0.000009  0.000133         6.445125  9.263004e-07   
 1     2013-01-01 01:00:00  0.000009  0.000141         5.901165  9.265574e-07   
 2     2013-01-01 02:00:00  0.000009  0.000146         5.779378  9.261187e-07   
 3     2013-01-01 03:00:00  0.000010  0.000779         4.329860  9.369592e-07   
 4     2013-01-01 04:00:00  0.000010  0.542396         3.242434  9.496740e-07   
 ...                   ...       ...       ...              ...           ...   
 8755  2013-12-31 19:00:00  0.000009  0.318752         7.144927  9.236718e-07   
 8756  2013-12-31 20:00:00  0.000010  0.213500         3.457981  9.357695e-07   
 8757  2013-12-31 21:00:00  0.000010  0.189965         3.772194  9.364199e-07   
 8758  2013-12-31 22:00:00  0.000010  0.096612         3.019120  9.355475e-07   
 8759  2013-12-31 23:00:00  0.000010  0.000279         3.373186  9.337951e-07   
 
       b

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [18]:
print (f"Name of the DispaSET Unleash folder:                          {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                          {dispaSET_unleash_folder_path}\n")
print (f"Name of the Availability Factors Base data folder:            {availability_factors_base_data_folder_name}\n")
print (f"Path to the Availability Factors Base data folder:            {availability_factors_base_data_folder_path}\n")
print (f"Name of Availability Factors_Pypsa Raw data folder:           {availability_factors_pypsa_raw_data_folder_name}\n")
print (f"Path to the Availability Factors_Pypsa Raw data folder:       {availability_factors_pypsa_raw_data_folder_path}\n")
print (f"Name of the Availability Factors_Pypsa Formated data folder:  {availability_factors_pypsa_formated_data_folder_name}\n")
print (f"Path to the Availability Factors_Pypsa Formated data folder:  {availability_factors_pypsa_formated_data_folder_path}\n")
print (f"Name of the Power Plants_Pypsa Formated data folder:          {power_plants_pypsa_formated_data_folder_name}\n")
print (f"Path to the Power Plants_Pypsa Formated data folder:          {power_plants_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                            {zone_names}\n")
print (f"Name of the zone_names_equivalences_dict (dictionary):        {list(zone_names_equivalences_dict.keys())}\n")
print (f"Target year:                                                  {data_target_year}\n")
print (f"Target time step:                                             {data_target_time_step}\n")
print (f"Name of the Availability Factors DataFrames (dictionary):     {list(availability_factors_dfs_dict.keys())}\n")
print (f"Name of the Technologies Equivalences Dictionary:             tech_equivalences_dict\n")
print (f"Name of the raw data DataFrames (dictionary):                 {list(nodal_dispatch_dfs_dict.keys())}\n")

Name of the DispaSET Unleash folder:                          Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                          /home/ray/Dispa-SET_Unleash

Name of the Availability Factors Base data folder:            AvailabilityFactors

Path to the Availability Factors Base data folder:            /home/ray/Dispa-SET_Unleash/Database/AvailabilityFactors

Name of Availability Factors_Pypsa Raw data folder:           dispatch

Path to the Availability Factors_Pypsa Raw data folder:       /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Suficiency_Scenario/dispatch

Name of the Availability Factors_Pypsa Formated data folder:  AvailabilityFactors

Path to the Availability Factors_Pypsa Formated data folder:  /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/AvailabilityFactors

Name of the Power Plants_Pypsa Formated data folder:          PowerPlants

Path to the Power Plants_Pypsa Formated data folder:          /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Extracting the requisite columns from the raw dataset for integration into the availability factors dataframes.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [19]:
# Process each zone
for zone in zone_names:
    # Get the dispatch DataFrame
    dispatch_df = nodal_dispatch_dfs_dict[f"{zone}_dispatch_df"]
    # Get the availability factors DataFrame
    availability_df = availability_factors_dfs_dict[f"{zone}_{data_target_year}"]

    # Iterate over tech_equivalences_dict
    for col_key, tech_info in tech_equivalences_dict.items():
        # Get the first element of the list for tech_col and curt_col
        tech_col = tech_info["tech"][0]  # e.g., "onshore wind"
        curt_col = tech_info["curt"][0] if tech_info["curt"][0].strip() else None  # Handle empty string as None

        # Check if the tech column exists in the dispatch DataFrame
        if tech_col in dispatch_df.columns:
            # Subtract curtailment if it exists and is not empty
            if curt_col and curt_col in dispatch_df.columns:
                availability_df[col_key] = dispatch_df[tech_col] - dispatch_df[curt_col]
            else:
                availability_df[col_key] = dispatch_df[tech_col]

# Now, availability_factors_dfs_dict contains the updated DataFrames

availability_factors_dfs_dict

{'BE_2030':                                     WTON      WTOF  PHOT      HROR
 0    2030-01-01 00:00:00+00:00  4.681409  5.313026   0.0  0.058519
 1    2030-01-01 01:00:00+00:00  4.591454  5.313026   0.0  0.058397
 2    2030-01-01 02:00:00+00:00  4.426489  5.313026   0.0  0.058159
 3    2030-01-01 03:00:00+00:00  4.216704  5.312708   0.0  0.058958
 4    2030-01-01 04:00:00+00:00  4.037236  5.291029   0.0  0.059017
 ...                        ...       ...       ...   ...       ...
 8755 2030-12-31 19:00:00+00:00  4.289843  5.313027   0.0 -0.499082
 8756 2030-12-31 20:00:00+00:00  4.050061  5.312882   0.0  0.043941
 8757 2030-12-31 21:00:00+00:00  3.750906  5.312500   0.0  0.043782
 8758 2030-12-31 22:00:00+00:00  3.469835  5.312701   0.0  0.043514
 8759 2030-12-31 23:00:00+00:00  3.300957  5.312816   0.0  0.043177
 
 [8760 rows x 5 columns],
 'FR_2030':                                      WTON       WTOF  PHOT      HROR
 0    2030-01-01 00:00:00+00:00  14.548289  23.027429   0.0  1.4

<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    8. PyPSA to Dispa-SET Availability Factors Data Formatting
    </div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The availability factor ($\text{AF}$) is defined as the proportion of the nominal power capacity that can be generated at each hour. To determine this factor from the available data (the actual power output), the nominal power capacity is essential as the reference maximum.
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: normal; font-size: 11px; font-family: TimesNewRoman; color:skyblue">
<ol style="margin-top: 0; padding-left: 1.5em;">
<li>
Kavvadias, K., Hidalgo Gonzalez, I., Zucker, A. and Quoilin, S., Integrated modelling of future EU power and heat systems: The Dispa-SET v2.2 open-source model, JRC Technical Report, EU Commission, 2018.
<br>
<a href="https://www.dispaset.eu/en/latest/data.html#unit-specific-or-technology-specific-inputs" style="color:skyblue">https://www.dispaset.eu/en/latest/data.html#unit-specific-or-technology-specific-inputs</a>
</li>
</ol>
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The availability factor is calculated using the following relationship:

$$\text{Availability Factor (AF)} = \frac{\text{Actual Power Output (Generation)}}{\text{Nominal Power Capacity (P}_{\text{nom}})}$$
</div>

<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Therefore, the nominal power capacity ($\text{P}_{\text{nom}}$) of the renewable unit is required as a divisor to calculate the availability factor time series.
</div>

<hr style="border: 0.5px solid skyblue;">
</div>

In [20]:
# Iterate over each zone
for zone in zone_names:
    # Get the availability factors DataFrame
    availability_df = availability_factors_dfs_dict[f"{zone}_{data_target_year}"]

    # Path to the power plants CSV file
    power_plants_file_path = os.path.join(
        power_plants_pypsa_formated_data_folder_path,
        zone,
        f"{data_target_year}.csv"
    )

    # Read the power plants CSV file
    power_plants_df = pd.read_csv(power_plants_file_path)

    # Iterate over each column in the availability factors DataFrame
    for column in availability_df.columns:
        # Find matching rows in the power plants DataFrame
        matching_rows = power_plants_df[power_plants_df['Technology'] == column]

        # Sum the PowerCapacity for matching rows
        total_power_capacity = matching_rows['PowerCapacity'].sum()

        # Avoid division by zero
        if total_power_capacity > 0:
            # Divide the column by the total PowerCapacity
            availability_df[column] = availability_df[column] / total_power_capacity

# Now, availability_factors_dfs_dict contains the updated DataFrames

availability_factors_dfs_dict

{'BE_2030':                                     WTON      WTOF  PHOT      HROR
 0    2030-01-01 00:00:00+00:00  0.000985  0.000886   0.0  0.000992
 1    2030-01-01 01:00:00+00:00  0.000966  0.000886   0.0  0.000990
 2    2030-01-01 02:00:00+00:00  0.000931  0.000886   0.0  0.000985
 3    2030-01-01 03:00:00+00:00  0.000887  0.000885   0.0  0.000999
 4    2030-01-01 04:00:00+00:00  0.000849  0.000882   0.0  0.001000
 ...                        ...       ...       ...   ...       ...
 8755 2030-12-31 19:00:00+00:00  0.000903  0.000886   0.0 -0.008457
 8756 2030-12-31 20:00:00+00:00  0.000852  0.000885   0.0  0.000745
 8757 2030-12-31 21:00:00+00:00  0.000789  0.000885   0.0  0.000742
 8758 2030-12-31 22:00:00+00:00  0.000730  0.000885   0.0  0.000737
 8759 2030-12-31 23:00:00+00:00  0.000694  0.000885   0.0  0.000732
 
 [8760 rows x 5 columns],
 'FR_2030':                                     WTON      WTOF  PHOT      HROR
 0    2030-01-01 00:00:00+00:00  0.000706  0.000860   0.0  0.00020

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Overall availability factors have been collected externally and will be used for comparative purposes, utilizing the maximum values at each time step.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [21]:
# Read the Overall_AF.csv file
overall_af_path = os.path.join(availability_factors_pypsa_raw_data_folder_path, "Overall_AF.csv")
overall_af_df = pd.read_csv(overall_af_path)

# Iterate over each key in availability_factors_dfs_dict
for key in availability_factors_dfs_dict:
    # Extract the zone acronym from the key (e.g., "BE" from "BE_2030")
    zone = key.split('_')[0]

    # Get the DataFrame
    availability_df = availability_factors_dfs_dict[key]

    # Initialize a dictionary to hold subgrouped columns
    subgrouped_columns = {tech_key: [] for tech_key in tech_equivalences_dict}

    # Find candidate columns for this zone
    candidate_columns = [col for col in overall_af_df.columns if zone in col]

    # For each candidate column, check if it matches any tech in tech_equivalences_dict
    for col in candidate_columns:
        for tech_key, tech_info in tech_equivalences_dict.items():
            for tech in tech_info["tech"]:
                if tech in col.lower():
                    subgrouped_columns[tech_key].append(col)
                    break

    # For each subgroup, select the column(s) to use
    for tech_key in subgrouped_columns:
        columns = subgrouped_columns[tech_key]
        if not columns:
            continue

        # Extract years from column names (if any)
        years = []
        year_columns = {}
        for col in columns:
            match = re.search(r'\d{4}', col)
            if match:
                year = int(match.group())
                years.append(year)
                if year not in year_columns:
                    year_columns[year] = []
                year_columns[year].append(col)

        # If years are found, select the columns with the closest year to data_target_year
        if years:
            closest_year = min(years, key=lambda y: abs(y - data_target_year))
            closest_cols = year_columns.get(closest_year, [])
            if closest_cols:
                # Take the maximum value per row across all columns with the closest year
                overall_col = overall_af_df[closest_cols].max(axis=1)
            else:
                continue
        else:
            # If no years are found, use all columns in the subgroup and take the max per row
            overall_col = overall_af_df[columns].max(axis=1)

        # Compare and update values in availability_df
        if len(overall_col) == len(availability_df[tech_key]):
            for i in range(len(overall_col)):
                if overall_col.iloc[i] > availability_df[tech_key].iloc[i]:
                    availability_df.at[i, tech_key] = overall_col.iloc[i]

# Now, availability_factors_dfs_dict contains the updated DataFrames
availability_factors_dfs_dict 

{'BE_2030':                                     WTON      WTOF  PHOT      HROR
 0    2030-01-01 00:00:00+00:00  0.984893  0.885500   0.0  1.000000
 1    2030-01-01 01:00:00+00:00  0.965968  0.885500   0.0  1.000000
 2    2030-01-01 02:00:00+00:00  0.931262  0.885500   0.0  1.000000
 3    2030-01-01 03:00:00+00:00  0.887126  0.885474   0.0  1.000000
 4    2030-01-01 04:00:00+00:00  0.849369  0.885115   0.0  1.000000
 ...                        ...       ...       ...   ...       ...
 8755 2030-12-31 19:00:00+00:00  0.902513  0.885500   0.0  0.748912
 8756 2030-12-31 20:00:00+00:00  0.852067  0.885500   0.0  0.745800
 8757 2030-12-31 21:00:00+00:00  0.789129  0.885500   0.0  0.743193
 8758 2030-12-31 22:00:00+00:00  0.729995  0.885500   0.0  0.738727
 8759 2030-12-31 23:00:00+00:00  0.694466  0.885500   0.0  0.733083
 
 [8760 rows x 5 columns],
 'FR_2030':                                     WTON      WTOF  PHOT      HROR
 0    2030-01-01 00:00:00+00:00  0.706225  0.883900   0.0  0.40093

<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
9. Final Formatted Data Storage
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Once the cleaned data has been obtained, it must be stored in the appropriate location within the Dispa-SET database file structure.
    <br>
A dedicated folder, named "Database_PyPSA", has been created to house all data related to the NEGAWAT project.
    <br>
This folder contains two primary subfolders corresponding to the project scenarios: "Reference_Scenario" and "Suficiency_Scenario".
        <br>
Each scenario subfolder contains nested sub-subfolders, each named using the acronym of a corresponding EU country.
        <br>
The processed and cleaned DataFrames are stored within their respective country sub-subfolders. Each DataFrame is converted into a .csv file and named after the processed year (e.g., `2030.csv`).
    </div>
    <hr style="border: 0.5px solid skyblue;">
</div>

In [22]:
# Iterate over each key in availability_factors_dfs_dict
for key in availability_factors_dfs_dict:
    # Extract the zone acronym from the key (e.g., "BE" from "BE_2030")
    zone = key.split('_')[0]

    # Construct the path to the zone folder
    zone_folder_path = os.path.join(availability_factors_pypsa_formated_data_folder_path, zone)

    # Check if the zone folder exists
    if not os.path.exists(zone_folder_path):
        print(f"Zone folder does not exist: {zone_folder_path}")
        continue

    # Construct the path to the data_target_time_step subfolder
    time_step_folder_path = os.path.join(zone_folder_path, data_target_time_step)

    # Create the data_target_time_step subfolder if it doesn't exist
    if not os.path.exists(time_step_folder_path):
        os.makedirs(time_step_folder_path)
        print(f"Created subfolder: {time_step_folder_path}")

    # Define the path for the CSV file
    csv_file_path = os.path.join(time_step_folder_path, f"{data_target_year}.csv")

    # Save the DataFrame to the CSV file
    availability_factors_dfs_dict[key].to_csv(csv_file_path, index=False)
    print(f"Saved DataFrame to: {csv_file_path}")

print("All DataFrames saved successfully!")

Saved DataFrame to: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/AvailabilityFactors/BE/1h/2030.csv
Saved DataFrame to: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/AvailabilityFactors/FR/1h/2030.csv
Saved DataFrame to: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/AvailabilityFactors/DE/1h/2030.csv
Saved DataFrame to: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/AvailabilityFactors/NL/1h/2030.csv
Saved DataFrame to: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/AvailabilityFactors/UK/1h/2030.csv
All DataFrames saved successfully!


<div style="background-color: black;">
<hr style="border: 4px solid skyblue;">
</div>